# Notebook 09 — Residual stream and logit-lens depth localization

Paired with `theory/09_residual_stream.md`. **Mode:** mixed.

## QHMPC

**Question.** The residual-stream picture predicts that intermediate computations are observable in the residual stream — and the logit lens, applied at depth $\ell$, reveals what next-token distribution the model would predict if it stopped reading at layer $\ell$. We test this on a **2-hop lookup task** that genuinely requires depth, so the logit-lens trajectory has structure: the *1-hop intermediate* should appear before the *2-hop answer*. 

Concretely:
1. Telescoping decomposition: norm of cumulative perturbation vs. depth at init.
2. Per-component contributions ($a^{(\ell)}$ vs. $m^{(\ell)}$).
3. **2-hop lookup task and logit-lens depth localization** — the load-bearing experiment.
4. Skip-connection survival: probe input token from final residual stream.
5. Layer ablation: does removing layer 1 break the first hop while leaving the second hop's mechanism in place?

**Hypothesis.** On a 2-hop lookup, a 4-layer model has to use its depth — one or two layers to do the first hop, one or two more for the second. Logit lens at layer 1 should resolve the 1-hop intermediate (e.g. predicting `b` given `a→b`); logit lens at layer 3-4 should resolve the 2-hop answer (predicting `c` given `a→b, b→c`). If logit lens shows the 2-hop answer at layer 1, the task isn't really 2-hop and the chapter's claim is weaker.

**Method.** Build a 4-layer transformer, train on the 2-hop lookup, instrument the residual stream and trace logit-lens predictions of *both* the 1-hop intermediate and the 2-hop answer at every depth.

**Prediction.** *Paste §9.7 predictions.*

**Confidence.** *[low / medium / high, reason]*.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

np.random.seed(0); torch.manual_seed(0)

## Setup — 2-hop lookup task and instrumented 4-layer model

**Task.** Vocabulary of $V$ tokens. A sequence consists of $K$ random `(key, value)` pairs followed by a query `q`. The pairs are constructed so that the value of *one* pair appears as a key in *another* pair, forming a chain `q → v_1 → v_2`. The model must output `v_2` (2-hop), not `v_1` (1-hop). All other pairs are distractors.

Example with $V = 8$ symbols `{a, b, c, d, e, f, g, h}` and $K = 4$ pairs: `[a, b, c, d, e, f, b, g, q=a]`. Hops: `a → b → g`. Target: `g`. The 1-hop intermediate `b` is also a meaningful prediction — it's the answer at half-depth.

In [ ]:
V = 16
K = 5            # number of (key, value) pairs in a sequence
SEQ_LEN = 2 * K + 1     # K pairs of (key, value) interleaved + the query
D_MODEL = 64
L = 4

def make_batch(B):
    """Build a batch with a guaranteed 2-hop chain. Returns (sequences, hop1_targets, hop2_targets)."""
    seqs = torch.zeros(B, SEQ_LEN, dtype=torch.long)
    hop1 = torch.zeros(B, dtype=torch.long)   # 1-hop answer (the intermediate)
    hop2 = torch.zeros(B, dtype=torch.long)   # 2-hop answer (the final target)
    for b in range(B):
        # Pick K distinct keys, K distinct values (overlap allowed between sets but no repeats within set).
        # Build chain: pair 0 has key q -> v1; pair 1 has key v1 -> v2.
        # Other K-2 pairs are random distractors with disjoint keys.
        symbols = np.random.permutation(V)
        q, v1, v2 = symbols[0], symbols[1], symbols[2]
        # Pick K-2 distractor pairs with keys not in {q, v1}
        other_keys = list(symbols[3:3 + (K - 2)])
        other_vals = list(np.random.choice(V, size=K - 2))
        # Ordered pairs to embed in the sequence (in random positions for non-positional robustness)
        pairs = [(int(q), int(v1)), (int(v1), int(v2))] + list(zip(other_keys, other_vals))
        np.random.shuffle(pairs)
        for i, (k_, v_) in enumerate(pairs):
            seqs[b, 2*i] = k_
            seqs[b, 2*i + 1] = v_
        seqs[b, -1] = int(q)
        hop1[b] = int(v1)
        hop2[b] = int(v2)
    return seqs, hop1, hop2

# sanity
s, h1, h2 = make_batch(2)
for b in range(2):
    print(f'seq: {s[b].tolist()}    query={s[b, -1].item()}    1-hop={h1[b].item()}    2-hop={h2[b].item()}')

In [ ]:
class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1 = nn.LayerNorm(D_MODEL); self.ln2 = nn.LayerNorm(D_MODEL)
        self.attn = nn.MultiheadAttention(D_MODEL, 4, batch_first=True, bias=False)
        self.ffn = nn.Sequential(nn.Linear(D_MODEL, 4 * D_MODEL), nn.GELU(), nn.Linear(4 * D_MODEL, D_MODEL))
    def forward(self, r):
        a, _ = self.attn(self.ln1(r), self.ln1(r), self.ln1(r), need_weights=False)
        a_only = a
        m_only = self.ffn(self.ln2(r + a_only))
        return r + a_only + m_only, a_only, m_only

class T(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(V, D_MODEL)
        self.pos = nn.Parameter(torch.randn(SEQ_LEN, D_MODEL) * 0.02)
        self.blocks = nn.ModuleList([Block() for _ in range(L)])
        self.ln_out = nn.LayerNorm(D_MODEL)
        self.unembed = nn.Linear(D_MODEL, V, bias=False)
    def forward(self, seq, return_trace=False):
        r = self.embed(seq) + self.pos
        trace = {'r': [r.detach().clone()], 'a': [], 'm': []}
        for blk in self.blocks:
            r, a, m = blk(r)
            if return_trace:
                trace['r'].append(r.detach().clone()); trace['a'].append(a.detach().clone()); trace['m'].append(m.detach().clone())
        logits = self.unembed(self.ln_out(r))
        return (logits[:, -1], trace) if return_trace else logits[:, -1]

torch.manual_seed(0)
model = T()
opt = torch.optim.Adam(model.parameters(), lr=3e-3)
for step in range(3000):
    seq, _, h2 = make_batch(64)
    loss = F.cross_entropy(model(seq), h2)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 500 == 0:
        with torch.no_grad():
            seq_e, _, h2_e = make_batch(500)
            acc = (model(seq_e).argmax(-1) == h2_e).float().mean().item()
            print(f'step {step}: loss = {loss.item():.4f}   acc(2-hop) = {acc:.3f}')

with torch.no_grad():
    seq_e, _, h2_e = make_batch(2000)
    print(f'\nfinal acc(2-hop) = {(model(seq_e).argmax(-1) == h2_e).float().mean().item():.3f}   (chance = 1/{V} = {1/V:.3f})')

## Experiment 1 — telescoping norm growth

In [ ]:
torch.manual_seed(1)
init_model = T()
seq, _, _ = make_batch(8)
with torch.no_grad():
    _, tr_init = init_model(seq, return_trace=True)
    _, tr_trained = model(seq, return_trace=True)

norms_init = [r.norm(dim=-1).mean().item() for r in tr_init['r']]
norms_trained = [r.norm(dim=-1).mean().item() for r in tr_trained['r']]

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(L + 1), norms_init, 'o-', label='at init')
ax.plot(range(L + 1), norms_trained, 's-', label='trained')
ax.set_xlabel('depth'); ax.set_ylabel('mean ||r^(L)||')
ax.set_title('Residual-stream norm vs. depth')
ax.legend(); plt.tight_layout(); plt.show()

**Sub-finding 1.** *Did norm grow as $\sqrt{L}$ at init? Did training change the growth pattern?*

## Experiment 2 — per-component contributions

In [ ]:
a_norms = [a.norm(dim=-1).mean().item() for a in tr_trained['a']]
m_norms = [m.norm(dim=-1).mean().item() for m in tr_trained['m']]

fig, ax = plt.subplots(figsize=(6, 4))
x = np.arange(L)
ax.bar(x - 0.2, a_norms, 0.4, label='||attn|| (a^l)')
ax.bar(x + 0.2, m_norms, 0.4, label='||ffn|| (m^l)')
ax.set_xlabel('layer'); ax.set_ylabel('mean norm')
ax.set_title('Per-layer contribution magnitudes (trained)')
ax.legend(); plt.tight_layout(); plt.show()

**Sub-finding 2.** *Did contributions concentrate in particular layers (which would suggest those layers do the load-bearing work)? Was attn comparable to ffn at each layer?*

---
## Experiment 3 — Logit-lens depth localization on the 2-hop task

At each depth $\ell$, apply `unembed(ln_out(r))` to the *query position*'s residual stream. Measure two accuracies:
- 1-hop accuracy: does the lens-projected prediction equal the *intermediate* `v_1`?
- 2-hop accuracy: does it equal the *final* `v_2`?

If the model uses its depth, we should see 1-hop accuracy rise *before* 2-hop accuracy.

In [ ]:
with torch.no_grad():
    seq, h1_t, h2_t = make_batch(2000)
    _, tr = model(seq, return_trace=True)
    acc_h1, acc_h2 = [], []
    for ell in range(L + 1):
        r = tr['r'][ell]
        logits_q = model.unembed(model.ln_out(r))[:, -1]
        preds = logits_q.argmax(-1)
        acc_h1.append((preds == h1_t).float().mean().item())
        acc_h2.append((preds == h2_t).float().mean().item())

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(L + 1), acc_h1, 'o-', label='predicts 1-hop intermediate (v_1)')
ax.plot(range(L + 1), acc_h2, 's-', label='predicts 2-hop answer (v_2)')
ax.axhline(1/V, color='k', linestyle=':', alpha=0.5, label=f'chance = {1/V:.3f}')
ax.set_xlabel('depth (logit lens applied here)')
ax.set_ylabel('accuracy')
ax.set_title('Logit lens reveals when each hop is computed')
ax.legend(); plt.tight_layout(); plt.show()

print(f'{"depth":>6}  {"1-hop acc":>12}  {"2-hop acc":>12}')
for ell in range(L + 1):
    print(f'{ell:>6}  {acc_h1[ell]:>12.3f}  {acc_h2[ell]:>12.3f}')

**Sub-finding 3.** *Did 1-hop accuracy rise BEFORE 2-hop accuracy across depths? At which layer did each peak? If 2-hop accuracy jumped at the same layer as 1-hop, the model isn't really doing 2-hop sequentially — it's somehow shortcutting. If 1-hop rises mid-stack and then DECAYS as 2-hop rises, the residual stream is overwriting the intermediate (interesting — it would weaken the "nothing is overwritten" claim).*

## Experiment 4 — input-token survival via linear probe

In [ ]:
X_data, y_data = [], []
model.eval()
with torch.no_grad():
    for _ in range(40):
        seq, _, _ = make_batch(32)
        _, tr = model(seq, return_trace=True)
        r_final = tr['r'][-1]
        X_data.append(r_final.reshape(-1, D_MODEL)); y_data.append(seq.reshape(-1))
X = torch.cat(X_data); y = torch.cat(y_data)

probe = nn.Linear(D_MODEL, V, bias=False)
opt2 = torch.optim.Adam(probe.parameters(), lr=1e-2)
for _ in range(500):
    loss = F.cross_entropy(probe(X), y)
    opt2.zero_grad(); loss.backward(); opt2.step()
with torch.no_grad():
    print(f'Linear probe acc on input token from r^(L): {(probe(X).argmax(-1) == y).float().mean().item():.3f}   (chance = {1/V:.3f})')

**Sub-finding 4.** *Did the input token survive 4 residual additions? At the query position specifically — does the original query token remain identifiable, or does it get overwritten by intermediate computation?*

## Experiment 5 — layer ablation on the 2-hop task

Skip one block (set $a^{(\ell)} + m^{(\ell)} = 0$); measure 1-hop and 2-hop accuracy separately. Hypothesis: ablating an early layer should kill the 1-hop computation and therefore both accuracies; ablating a late layer should kill 2-hop while leaving 1-hop intact in the lens.

In [ ]:
def ablate_layer(model, layer_idx, seq):
    r = model.embed(seq) + model.pos
    for ell, blk in enumerate(model.blocks):
        r_new, a, m = blk(r)
        if ell != layer_idx:
            r = r_new
    return model.unembed(model.ln_out(r))[:, -1]

with torch.no_grad():
    seq, h1_t, h2_t = make_batch(1000)
    base_pred = model(seq).argmax(-1)
    base_h1 = (base_pred == h1_t).float().mean().item()
    base_h2 = (base_pred == h2_t).float().mean().item()
    print(f'baseline:           1-hop = {base_h1:.3f}   2-hop = {base_h2:.3f}')
    print()
    for ell in range(L):
        preds = ablate_layer(model, ell, seq).argmax(-1)
        a_h1 = (preds == h1_t).float().mean().item()
        a_h2 = (preds == h2_t).float().mean().item()
        print(f'ablate layer {ell}:   1-hop = {a_h1:.3f} ({a_h1 - base_h1:+.3f})   2-hop = {a_h2:.3f} ({a_h2 - base_h2:+.3f})')

**Sub-finding 5.** *Did early-layer ablation hurt both hops? Did late-layer ablation hurt 2-hop more than 1-hop? Is there a clean depth assignment of "layer X does hop 1, layer Y does hop 2," or is the computation distributed?*

## Finding

*3–6 sentences. Did the logit lens cleanly separate the 1-hop intermediate from the 2-hop answer across depth? If yes, the residual stream picture is empirically validated for *this* class of multi-step tasks. If the 1-hop signal didn't appear in the lens at any depth, either (a) the model found a shortcut that doesn't pass through the intermediate, or (b) the intermediate is encoded in a subspace orthogonal to the unembedding's columns. Both are interesting updates to the mental model.*